### Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Base Paths
SILVER_BASE_PATH = "abfss://silver@stretaillakehouse01.dfs.core.windows.net/"
GOLD_BASE_PATH   = "abfss://gold@stretaillakehouse01.dfs.core.windows.net/"

# Source Path
SILVER_STORE_PATH = SILVER_BASE_PATH + "stores/"

# Target Path
GOLD_DIM_STORE_PATH = GOLD_BASE_PATH + "dim_store/"

### Reading from Silver Layer

In [0]:
df_dim_store = (
    spark.read
         .format("parquet")
         .load(SILVER_STORE_PATH)
)

In [0]:
df_dim_store.printSchema()

df_dim_store.show(5, truncate=False)

print(f"Total Rows: {df_dim_store.count()}")

root
 |-- Store_ID: string (nullable = true)
 |-- Store_Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Employee_Count: integer (nullable = true)
 |-- Opening_Date: date (nullable = true)

+--------+---------------+-------+-----------+------+----------+--------------+------------+
|Store_ID|Store_Name     |Country|State      |City  |Store_Type|Employee_Count|Opening_Date|
+--------+---------------+-------+-----------+------+----------+--------------+------------+
|S001    |Pune Retail 1  |India  |Maharashtra|Pune  |Outlet    |81            |2025-03-06  |
|S002    |Nashik Retail 2|India  |Maharashtra|Nashik|Standalone|84            |2024-08-22  |
|S003    |Nashik Retail 3|India  |Maharashtra|Nashik|Standalone|59            |2020-06-02  |
|S004    |Pune Retail 4  |India  |Maharashtra|Pune  |Outlet    |86            |2018-01-19  |
|S005    |Mumbai R

### Null Value Evaluation

In [0]:
df_dim_store.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_dim_store.columns
    ]
).show()

+--------+----------+-------+-----+----+----------+--------------+------------+
|Store_ID|Store_Name|Country|State|City|Store_Type|Employee_Count|Opening_Date|
+--------+----------+-------+-----+----+----------+--------------+------------+
|       0|         0|      0|    0|   0|         0|             0|           0|
+--------+----------+-------+-----+----+----------+--------------+------------+



### Duplicate Value Evaluation

In [0]:
(
    df_dim_store
    .groupBy("Store_ID")
    .count()
    .filter(col("count") > 1)
    .show()
)

+--------+-----+
|Store_ID|count|
+--------+-----+
+--------+-----+



### Writing to Gold Layer

In [0]:
(
    df_dim_store.write
    .mode("overwrite")
    .format("parquet")
    .save(GOLD_DIM_STORE_PATH)
)

### Reading from Gold Layer

In [0]:
df_gold_dim_store = (
    spark.read
         .format("parquet")
         .load(GOLD_DIM_STORE_PATH)
)

### Validation

In [0]:
df_gold_dim_store.printSchema()

df_gold_dim_store.show(10, truncate=False)

print(f"Gold Row Count: {df_gold_dim_store.count()}")

root
 |-- Store_ID: string (nullable = true)
 |-- Store_Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Employee_Count: integer (nullable = true)
 |-- Opening_Date: date (nullable = true)

+--------+---------------+-------+-----------+------+----------+--------------+------------+
|Store_ID|Store_Name     |Country|State      |City  |Store_Type|Employee_Count|Opening_Date|
+--------+---------------+-------+-----------+------+----------+--------------+------------+
|S001    |Pune Retail 1  |India  |Maharashtra|Pune  |Outlet    |81            |2025-03-06  |
|S002    |Nashik Retail 2|India  |Maharashtra|Nashik|Standalone|84            |2024-08-22  |
|S003    |Nashik Retail 3|India  |Maharashtra|Nashik|Standalone|59            |2020-06-02  |
|S004    |Pune Retail 4  |India  |Maharashtra|Pune  |Outlet    |86            |2018-01-19  |
|S005    |Mumbai R